In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from fundus_data_toolkit.functional import open_image
from jppype import Mosaic, vscode_theme

from fundus_odmac_toolkit.models.segmentation import segment
from fundus_toolkits import FundusData
from fundus_vessels_toolkit import VTree
from fundus_vessels_toolkit.models import segment_av
from fundus_vessels_toolkit.pipelines.avseg_to_tree import GNNAVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TreeTopology, optimal_lines
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from fundus_vessels_toolkit.utils.jppype import draw_graph, draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

## Load Image and Segment AV, OD, Macula


In [3]:
PATH = Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/MAPLES-DR/")
RAW = PATH / "1-images"
AV = PATH / "2-av"
TOPO = PATH / "3-topo"
IMG = sorted(list(AV.glob("*.png")))[25].stem  # 18

fundus_gt = FundusData(image=RAW / (IMG + ".png"), av=AV / (IMG + ".png"))
trees_gt = VTree.load(TOPO / f"{IMG}_art.npz"), VTree.load(TOPO / f"{IMG}_vei.npz")
topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=5, sparse=False, discard_tree=True),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=5, sparse=False, discard_tree=True),
)
sparse_topo_gt = (
    TreeTopology.from_tree(trees_gt[0], expand_labels_by=5, sparse=True, discard_tree=True),
    TreeTopology.from_tree(trees_gt[1], expand_labels_by=5, sparse=True, discard_tree=True),
)

od_mac = segment(open_image(RAW / (IMG + ".png"))).numpy(force=True).argmax(axis=0)
fundus_gt = fundus_gt.update(od=od_mac == 1, macula=od_mac == 2, reshape_method="resize")
fundus = fundus_gt.copy()
_ = segment_av(fundus)

av2tree = GNNAVSegToTree()
graph = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()

print(IMG)

20051202_51488_0400_PP


In [4]:
from fundus_vessels_toolkit.segment_to_graph.tree_topology import TopologicalLabel


def draw_topology(topo: TreeTopology, img=None):
    if img is None:
        img = fundus.image.transpose(1, 2, 0) * 0.5
    color_map = np.zeros(topo.shape + (3,), dtype=np.float32)
    alpha = np.zeros(topo.shape, dtype=np.float32)

    subtree_map = TopologicalLabel.decode_subtree(topo.branch_map)
    N_subtree = int(subtree_map.max()) + 1

    for s in range(0, N_subtree):
        mask = subtree_map == s
        if mask.sum() == 0:
            continue
        color_map[mask] = TopologicalLabel.subtree_color(s, format="rgb") / 255.0
        subtree_topo = topo.rank_map[mask]
        alpha[mask] = 1 - 0.8 * (np.floor(subtree_topo) + subtree_topo) / (subtree_topo.max() * 2)

    alpha = alpha[:, :, None]

    return (1 - alpha) * img + alpha * color_map


def draw_topos(topos):
    img = fundus.image.transpose(1, 2, 0) * 0.5
    img = draw_topology(topos[0], img)
    img = draw_topology(topos[1], img)
    return img

In [5]:
digraph = VBranchDigraph.from_graph(graph, max_distance=200, max_angle=45)
line_p = digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])
assert np.all(line_p == digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1])), (
    "Line probabilities from sparse and dense topology should be identical."
)

solved_tree = digraph.optimize_tree(keep_invalid_branch=True)

m = Mosaic(
    3, cols_titles=["Predicted", "Predicted with GT Topology", "Ground Truth"], cell_height=800, background=fundus.image
)
fundus.draw(view=m[0])
draw_graph(digraph.graph, view=m[0], edge_labels=True, node_labels=True)
m[1].add_image(fundus.image, name="fundus")
# m[1].add_image(draw_topology(topo_gt[0]), name="topos")
# draw_graph(sol, view=m[1])
fundus.draw(view=m[1])
draw_tree(
    solved_tree,
    view=m[1],
    branch_color="subtree",
    bspline_dir=True,
)

fundus_gt.draw(view=m[2])
# m[2].add_image(draw_topology(topo_gt[0]), name="topo")
# m[2].add_image(
#     np.stack(
#         [
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].fuzzy_skeleton_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
# m[2].add_image(
#     np.stack(
#         [
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             np.zeros_like(topo_gt[0].fuzzy_skeleton_map),
#             topo_gt[1].rank_map,
#         ]
#     ).transpose(1, 2, 0),
#     name="skeleton",
#     opacity=0.9,
# )
draw_trees(trees_gt, view=m[2], bspline_dir=True)  # , edge="skeleton")
m


GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

In [6]:
VBranchDigraph.from_graph(graph, max_distance=200, max_angle=45)

In [7]:
digraph.compute_p_from_gt(topo_gt[0], topo_gt[1])

array([ True,  True, False, ..., False, False, False])

In [9]:
from fundus_vessels_toolkit.segment_to_graph.tree_topology import read_branch_topology


label1, dir1, plau1, tip_label1, tip_rank1 = read_branch_topology(graph, topo_gt[0])
label2, dir2, plau2, tip_label2, tip_rank2 = read_branch_topology(graph, sparse_topo_gt[0])
assert np.all(label1 == label2), "Branch labels from sparse and dense topology should be identical."
assert np.all(np.isclose(dir1, dir2)), "Branch directions from sparse and dense topology should be identical."
assert np.all(tip_label1 == tip_label2), "Tip labels from sparse and dense topology should be identical."
assert np.all(np.isclose(tip_rank1, tip_rank2)), "Tip ranks from sparse and dense topology should be identical."
assert np.all(np.isclose(plau1, plau2, atol=1e-4)), (
    "Branch plausibility from sparse and dense topology should be identical."
)


AssertionError: Branch labels from sparse and dense topology should be identical.

In [ ]:
np.stack([tip_rank1, tip_rank2], axis=1)[~np.isclose(tip_rank1, tip_rank2, atol=1e-4)]
np.argwhere(~np.isclose(tip_rank1, tip_rank2, atol=1e-4).all(axis=1))

array([], shape=(0, 1), dtype=int64)

In [14]:
np.stack([np.arange(len(plau1)), plau1, plau2, label1, label2], axis=1)[~np.isclose(plau1, plau2, atol=1e-4)]


array([[7.00000000e+00, 8.79324794e-01, 0.00000000e+00, 6.19244949e+15,
        0.00000000e+00],
       [3.40000000e+01, 8.42835307e-01, 0.00000000e+00, 6.19244949e+15,
        0.00000000e+00],
       [4.50000000e+01, 9.73963141e-01, 0.00000000e+00, 5.06654958e+15,
        0.00000000e+00]])

In [ ]:
%timeit VBranchDigraph.from_graph(graph, max_distance=200)
%timeit digraph.compute_p_from_gt(sparse_topo_gt[0], sparse_topo_gt[1])
%timeit digraph.optimize_tree(keep_invalid_branch=True)

197 ms ± 8.46 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
41.2 ms ± 2.66 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
202 ms ± 1.05 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
from fundus_vessels_toolkit.segment_to_graph.geometry_parsing import derive_tips_geometry_from_curve_geometry
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import find_facing_tips
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import prepare_graph_for_reconnections

max_distance = 200
max_angle = 30
tan_max_angle = 80
pos_tolerance = 25
%timeit prepare_graph_for_reconnections(graph, max_distance=max_distance, max_angle=max_angle, av_attr="av", inplace=False)
_, candidates = prepare_graph_for_reconnections(
    graph, max_distance=max_distance, max_angle=max_angle, av_attr="av", inplace=True
)

%timeit derive_tips_geometry_from_curve_geometry(graph, tangent=True, inplace=False)
derive_tips_geometry_from_curve_geometry(graph, tangent=True, inplace=True)

%timeit find_facing_tips(graph,max_distance=max_distance,max_angle=max_angle,tan_max_angle=tan_max_angle,pos_tolerance=pos_tolerance,as_mask=True,)
facing_tips = find_facing_tips(
    graph,
    max_distance=max_distance,
    max_angle=max_angle,
    tan_max_angle=tan_max_angle,
    pos_tolerance=pos_tolerance,
    as_mask=True,
)
for b0, tip0, n1 in candidates:
    node = graph.node(n1)
    b1 = np.array(node.adjacent_branch_ids)
    tip1 = np.where(node.adjacent_branches_first_node, 0, 1)
    facing_tips[b0, tip0, b1, tip1] = True
    facing_tips[b1, tip1, b0, tip0] = True

%timeit graph.branch_tips_connectivity_matrix()
%timeit np.argwhere(facing_tips)


116 ms ± 910 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
18.4 ms ± 75.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
4.84 ms ± 28.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
93.7 µs ± 11 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
1.86 ms ± 46.1 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [ ]:
from fundus_vessels_toolkit.segment_to_graph.graph_simplification import find_reconnection_candidates
from fundus_vessels_toolkit.utils.cluster import cluster_by_distance
from fundus_vessels_toolkit.utils.math import softmax
from fundus_vessels_toolkit.utils.numpy import np_group_by

from fundus_vessels_toolkit.vascular_data_objects.vbranch_geodata import VBranchGeoData

g = av2tree.to_vgraph(fundus).sort_branches_by_nodesID()
g.geometric_data().clear_attribute(all_except={VBranchGeoData.Fields.TANGENTS, VBranchGeoData.Fields.TIPS_TANGENT})
max_distance = 100
max_angle = 30
snap_tip_max_distance = 30
snap_tip_max_angle = 30
snap_new_node_max_distance = 25


print("=== GET ENDPOINTS ===")
%timeit get_endpoints(g, av_attr="av")
endpoints = get_endpoints(g, av_attr="av")

print("=== FIND RECONNECTION CANDIDATES ===")
%timeit find_reconnection_candidates(g, max_distance=max_distance, max_angle=max_angle, snap_max_distance=snap_tip_max_distance, snap_max_angle=snap_tip_max_angle, endpoint_ids=endpoints)
candidates = find_reconnection_candidates(
    g,
    max_distance=max_distance,
    max_angle=max_angle,
    snap_max_distance=snap_tip_max_distance,
    snap_max_angle=snap_tip_max_angle,
    endpoint_ids=endpoints,
)

# Candidates format:  0     1   2          3         4  5  6
#                   (b0, tip0, n1, branch_id, curve_id, y, x)
new_node_mask = candidates[:, 2] == -1  # Node2 is a new node
reconnections = [candidates[~new_node_mask][:, :3]]
new_nodes_candidates = candidates[new_node_mask]


def snap_new_nodes():
    # Snap new nodes of the same branch if they are close enough
    nodes_yx = g.geometric_data().node_coord()
    nodes_specs = new_nodes_candidates[:, 2:]
    CANDIDATE, N1_BRANCH, N1_CURVE_ID, N1_YX = 0, 1, 2, slice(3, 5)
    nodes_specs[:, CANDIDATE] = np.arange(len(nodes_specs))
    new_nodes_lookup = np.full(len(new_nodes_candidates), -1, dtype=np.int_)

    merged_nodes_specs = []
    for b_id, branch_specs in np_group_by(nodes_specs, keys=nodes_specs[:, N1_BRANCH]):
        if len(branch_specs) == 1:
            new_id = len(merged_nodes_specs)
            merged_nodes_specs.append([new_id, *branch_specs[0, 1:]])
            new_nodes_lookup[branch_specs[:, CANDIDATE]] = new_id
            continue

        clusters = cluster_by_distance(branch_specs[:, N1_YX], snap_new_node_max_distance)
        for c in clusters:
            cluster_specs = branch_specs[c]
            new_id = len(merged_nodes_specs)
            if len(c) == 1:
                merged_nodes_specs.append([new_id, *cluster_specs[0, 1:]])
            else:
                b0, b0tip = new_nodes_candidates[cluster_specs[:, CANDIDATE], :2].T
                n0 = g.branch_list[b0, b0tip]
                sqr_dist = np.square(nodes_yx[n0] - cluster_specs[:, N1_YX]).sum(axis=1)
                n0_weight = softmax(-sqr_dist * 1e-3)
                centroid_yx = (cluster_specs[:, N1_YX] * n0_weight[:, None]).sum(axis=0)
                centroid_i = np.round((cluster_specs[:, N1_CURVE_ID] * n0_weight).sum(axis=0)).astype(np.int_)

                merged_nodes_specs += [(new_id, b_id, centroid_i, *centroid_yx)]
            new_nodes_lookup[cluster_specs[:, CANDIDATE]] = new_id
    return np.array(merged_nodes_specs), new_nodes_lookup


print("=== SNAP NEW NODES ===")
%timeit snap_new_nodes()
new_nodes_specs, new_nodes_lookup = snap_new_nodes()

LOOKUP_KEY, NEW_NODE_BRANCH, NEW_NODE_CURVE_ID, NEW_NODE_YX = 0, 1, 2, slice(3, 5)

branch_last_tip_lookup = np.arange(graph.branch_count)
# Split the branches at the new nodes
branch_ids, node_specs = zip(*np_group_by(new_nodes_specs, new_nodes_specs[:, NEW_NODE_BRANCH]), strict=True)


def split_graph():
    graph = g.copy()
    reconnections = []
    for b, branch_specs in zip(graph.branches(branch_ids, dynamic_iterator=True), node_specs, strict=True):
        if len(branch_specs) == 0:
            continue
        branch_specs = branch_specs[np.argsort(branch_specs[:, NEW_NODE_CURVE_ID])]  # Sort by curve index
        _, new_branch_id, new_nodes_id = graph.split_branch(
            branch_id=b.id,
            split_curve_id=branch_specs[:, NEW_NODE_CURVE_ID],
            split_coord=branch_specs[:, NEW_NODE_YX],
            return_node_ids=True,
            return_branch_ids=True,
            inplace=True,
        )
        branch_last_tip_lookup[b.id] = new_branch_id[-1]
        for lookup_key, new_node_id in zip(branch_specs[:, LOOKUP_KEY], new_nodes_id, strict=True):
            b0_b0tip = new_nodes_candidates[new_nodes_lookup == lookup_key][:, :2]
            assert len(b0_b0tip) > 0, "Lookup error for new node reconnection"
            reconnections += [np.hstack([b0_b0tip, np.full((len(b0_b0tip), 1), new_node_id)])]
    return graph, np.vstack(reconnections)


print("=== SPLIT GRAPH ===")
%timeit split_graph()
g, reconnections = split_graph()
last_tip_recon = reconnections[:, 1] == 1
reconnections[last_tip_recon, 0] = branch_last_tip_lookup[reconnections[last_tip_recon, 0]]


=== GET ENDPOINTS ===


NameError: name 'get_endpoints' is not defined

In [ ]:
get_endpoints_legacy(graph, av_attr="av")

array([[159,   0],
       [185,   0],
       [134,   0],
       [133,   0],
       [130,   0],
       [ 29,   0],
       [ 55,   0],
       [ 56,   0],
       [ 23,   0],
       [ 23,   0],
       [ 23,   1],
       [ 23,   1],
       [204,   0],
       [116,   0],
       [104,   0],
       [ 12,   0],
       [128,   0],
       [ 29,   1],
       [141,   0],
       [201,   0],
       [ 30,   1],
       [204,   1],
       [167,   0],
       [170,   0],
       [201,   1],
       [ 94,   0],
       [ 91,   0],
       [140,   0],
       [138,   0],
       [125,   0],
       [171,   1],
       [ 74,   0],
       [133,   1],
       [131,   1],
       [ 13,   0],
       [152,   0],
       [182,   0],
       [160,   1],
       [ 72,   0],
       [151,   0],
       [ 40,   0],
       [ 39,   0],
       [ 38,   0],
       [143,   0],
       [197,   0],
       [197,   1],
       [182,   1],
       [183,   1],
       [187,   0],
       [ 16,   0],
       [187,   1],
       [165,   0],
       [  3,

In [ ]:
get_endpoints(graph, av_attr="av")

array([[  5,   0],
       [  5,   1],
       [  6,   0],
       [  6,   1],
       [  7,   1],
       [ 10,   1],
       [ 12,   0],
       [ 12,   1],
       [ 16,   1],
       [ 17,   0],
       [ 18,   0],
       [ 19,   0],
       [ 19,   1],
       [ 20,   0],
       [ 24,   0],
       [ 27,   0],
       [ 32,   0],
       [ 32,   1],
       [ 33,   0],
       [ 33,   1],
       [ 35,   1],
       [ 37,   1],
       [ 39,   1],
       [ 45,   0],
       [ 46,   1],
       [ 48,   1],
       [ 49,   1],
       [ 51,   1],
       [ 53,   1],
       [ 56,   0],
       [ 57,   0],
       [ 57,   1],
       [ 58,   1],
       [ 59,   1],
       [ 60,   0],
       [ 60,   1],
       [ 61,   0],
       [ 62,   0],
       [ 63,   1],
       [ 64,   0],
       [ 64,   1],
       [ 65,   1],
       [ 66,   0],
       [ 66,   1],
       [ 67,   0],
       [ 69,   1],
       [ 70,   1],
       [ 71,   1],
       [ 72,   0],
       [ 73,   0],
       [ 74,   0],
       [ 76,   1],
       [ 77,

(array([  3. ,   3.5,  13. ,  14.5,  16. ,  18.5,  22. ,  23. ,  23.5,
         29.5,  30.5,  31. ,  38. ,  39. ,  43.5,  50. ,  52.5,  55. ,
         70. ,  77.5,  83.5,  85.5,  86.5,  88.5,  96. , 104. , 105.5,
        107.5, 108.5, 110.5, 116. , 117.5, 118.5, 121. , 121.5, 122.5,
        124.5, 125. , 127. , 128. , 128.5, 130. , 130.5, 131.5, 132.5,
        133. , 133.5, 134. , 136.5, 137.5, 138. , 138.5, 140. , 141. ,
        142.5, 143. , 144.5, 145.5, 146. , 146.5, 147. , 148.5, 149. ,
        149.5, 151. , 151.5, 152. , 154. , 155.5, 156.5, 157.5, 159. ,
        160.5, 164. , 165. , 167. , 170. , 171.5, 172. , 173. , 175. ,
        175.5, 176.5, 178.5, 179.5, 182. , 182.5, 183.5, 184. , 184.5,
        185. , 186.5, 187. , 187.5, 192. , 195. , 196. , 196.5, 197. ,
        197.5, 198. , 198.5, 200. , 201. , 201.5, 202.5, 203.5, 204. ,
        204.5, 205. , 206. , 207. , 207.5, 208.5, 209. , 210. ]),
 array([  1. ,   1.5,   4.5,   5. ,   5.5,   6. ,   6.5,   8. ,   9. ,
         11

In [ ]:
def draw_cone(branch_id: int, first_tip: bool, view=None, pos_tolerance=15, max_dist=100, max_angle=30):
    sqr_max_dist = max_dist * max_dist
    sqr_pos_tolerance = pos_tolerance * pos_tolerance

    min_cos = np.cos(np.deg2rad(max_angle))

    geodata = graph.geometric_data()
    tips_pos = (
        geodata.tip_coord(branch_id=branch_id, first_tip=first_tip).astype(np.float64).reshape(-1, 2)
    )  # [branch_id x (tip0, tip1), (y,x)]
    tips_tan = geodata.tip_tangent(branch_id=branch_id, first_tip=first_tip).reshape(
        -1, 2
    )  # [branch_id x (tip0, tip1), (y,x)]

    yy, xx = np.meshgrid(np.arange(fundus.image.shape[1]), np.arange(fundus.image.shape[2]), indexing="ij")
    yx = np.stack((yy, xx), axis=-1).reshape(-1, 2)

    tips_dtan = tips_pos[:, None, :] - yx[None, :, :]  # (tip_origin, tip_destination, yx)
    tips_dsqr = np.square(tips_dtan).sum(axis=2)
    tips_dtan /= np.sqrt(tips_dsqr)[..., None] + 1e-8

    # === VICINITY CHECK ===
    # Given a tip p0 with tangent t0 (oriented towards its curve)
    # we define a cone oriented towards -t0 with apex at p0 + t0 * pos_tolerance (so the tip itself is inside the cone)
    # and opening angle max_angle at distance pos_tolerance and 60 degrees at distance 0 from the apex.
    apex = tips_pos + tips_tan * pos_tolerance  # Cone apex position
    apex2tips = apex[:, None, :] - yx[None, :, :]
    apex2tips_dsqr = np.square(apex2tips).sum(axis=2)
    apex2tips /= np.sqrt(apex2tips_dsqr)[..., None] + 1e-8
    apex_cos = (tips_tan[:, None, :] * apex2tips).sum(axis=2)
    inside_cone = ((apex_cos >= min_cos) | (tips_dsqr <= sqr_pos_tolerance)) & (tips_dsqr <= sqr_max_dist)
    yx = yx[inside_cone[0]]
    map = np.zeros(fundus.image.shape[1:], dtype=np.uint8)
    map[yx[:, 0], yx[:, 1]] = 1
    if view is not None:
        view.add_label(map, "cone", opacity=0.2)